In [20]:

import pandas as pd
from pathlib import Path

# Identify the project directory
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

# Locate the training dataset
training_path = project_root / "Data" / "Raw data" / "UNSW_NB15_training-set.csv"

# Load the dataset
df_train = pd.read_csv(training_path)

# Verify the dataset
print("Training dataset shape:", df_train.shape)
display(df_train.head())

Training dataset shape: (175341, 45)


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,...,1,1,0,0,0,1,1,0,Normal,0
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,...,1,2,0,0,0,1,6,0,Normal,0
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,...,1,3,0,0,0,2,6,0,Normal,0
3,4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,...,1,3,1,1,0,2,1,0,Normal,0
4,5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,...,1,40,0,0,0,2,39,0,Normal,0


In [21]:

# Assess training dataset quality

print("Dataset shape:", df_train.shape)

print("\nMissing values:", df_train.isna().sum().sum())

print("\nDuplicate records (excluding ID):")
print(df_train.drop(columns=["id"]).duplicated().sum())

print("\nTarget distribution:")
print(df_train["label"].value_counts())

print("\nColumn data types:")
print(df_train.dtypes.value_counts())


Dataset shape: (175341, 45)

Missing values: 0

Duplicate records (excluding ID):
67601

Target distribution:
label
1    119341
0     56000
Name: count, dtype: int64

Column data types:
int64      30
float64    11
str         4
Name: count, dtype: int64


In [22]:

# Identify duplicates, excluding the unique ID
duplicate_mask = df_train.drop(columns=["id"]).duplicated()

# Calculate the proportion of duplicates
duplicate_percentage = duplicate_mask.mean() * 100

print(f"Duplicate percentage: {duplicate_percentage:.2f}%")

# Check duplicates by traffic class
print("\nDuplicates by traffic class:")
print(df_train.loc[duplicate_mask, "label"].value_counts().sort_index())

# Count unique records
print("\nUnique records:", (~duplicate_mask).sum())


Duplicate percentage: 38.55%

Duplicates by traffic class:
label
0     4110
1    63491
Name: count, dtype: int64

Unique records: 107740


In [23]:

# Select the features intended for Model Training
feature_columns = [
    col for col in df_train.columns
    if col not in ["id", "attack_cat", "label"]
]

# Find repeated feature combinations
feature_groups = df_train.groupby(
    feature_columns,
    dropna=False
)["label"].nunique()

# Count combinations associated with both labels
conflicting_groups = (feature_groups > 1).sum()

print("Input features:", len(feature_columns))
print("Feature combinations with conflicting labels:", conflicting_groups)



Input features: 42
Feature combinations with conflicting labels: 229


In [24]:

# Group records by their 42 input features
feature_groups = df_train.groupby(
    feature_columns,
    dropna=False
)["label"].agg(["nunique", "size"])

# Select groups containing conflicting labels
conflicting_groups = feature_groups[
    feature_groups["nunique"] > 1
]

# Measure the impact
print("Conflicting groups:", len(conflicting_groups))

print(
    "Total records in conflicting groups:",
    conflicting_groups["size"].sum()
)

print(
    "Largest conflicting group:",
    conflicting_groups["size"].max()
)


Conflicting groups: 229
Total records in conflicting groups: 940
Largest conflicting group: 23


In [25]:

# Create a working copy without changing the original dataset
df_clean = df_train.copy()

# Remove columns excluded from Binary Classification
df_clean = df_clean.drop(columns=["id", "attack_cat"])

# Verify the resulting DataFrame
print("Original dataset:", df_train.shape)
print("Working dataset:", df_clean.shape)

print("\nTarget column preserved:")
print("label" in df_clean.columns)

Original dataset: (175341, 45)
Working dataset: (175341, 43)

Target column preserved:
True


In [26]:

# Identify categorical features
categorical_columns = df_clean.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Categorical columns:", categorical_columns)

# Inspect the number of unique categories
for col in categorical_columns:
    print(f"\nColumn: {col}")
    print("Unique categories:", df_clean[col].nunique())
    print(df_clean[col].value_counts(dropna=False).head(5))


Categorical columns: ['proto', 'service', 'state']

Column: proto
Unique categories: 133
proto
tcp     79946
udp     63283
unas    12084
arp      2859
ospf     2595
Name: count, dtype: int64

Column: service
Unique categories: 13
service
-           94168
dns         47294
http        18724
smtp         5058
ftp-data     3995
Name: count, dtype: int64

Column: state
Unique categories: 9
state
INT    82275
FIN    77825
CON    13152
REQ     1991
RST       83
Name: count, dtype: int64


In [27]:

# Check placeholder values in categorical features

for col in categorical_columns:
    values = df_clean[col].astype("string").str.strip()

    print(f"\nFeature: {col}")
    print("Dash (-) values:", values.eq("-").sum())
    print("Empty strings:", values.eq("").sum())
    print("Missing values:", values.isna().sum())

# Display all connection states
print("\nAll state categories:")
print(df_clean["state"].value_counts().to_string())



Feature: proto
Dash (-) values: 0
Empty strings: 0
Missing values: 0

Feature: service
Dash (-) values: 94168
Empty strings: 0
Missing values: 0

Feature: state
Dash (-) values: 0
Empty strings: 0
Missing values: 0

All state categories:
state
INT    82275
FIN    77825
CON    13152
REQ     1991
RST       83
ECO       12
PAR        1
URN        1
no         1


In [28]:

import numpy as np

# Select numerical input features
numerical_columns = df_clean.select_dtypes(
    include=["number"]
).columns.drop("label")

print("Numerical features:", len(numerical_columns))

# Check infinite values
infinite_values = np.isinf(
    df_clean[numerical_columns].to_numpy()
).sum()

print("Infinite values:", infinite_values)

# Check negative values
negative_counts = (df_clean[numerical_columns] < 0).sum()

print("\nFeatures containing negative values:")
print(negative_counts[negative_counts > 0])

# Identify constant features
constant_columns = [
    col for col in numerical_columns
    if df_clean[col].nunique() <= 1
]

print("\nConstant features:", constant_columns)


Numerical features: 39
Infinite values: 0

Features containing negative values:
Series([], dtype: int64)

Constant features: []


In [29]:

# Load the official Test Dataset
test_path = project_root / "Data" / "Raw data" / "UNSW_NB15_testing-set.csv"
df_test = pd.read_csv(test_path)

# Select the same 42 input features in both datasets
input_features = df_clean.drop(columns=["label"]).columns.tolist()

# Create a fingerprint for each feature combination
train_hashes = pd.util.hash_pandas_object(
    df_clean[input_features], index=False
)

test_hashes = pd.util.hash_pandas_object(
    df_test[input_features], index=False
)

# Identify test records with matching training features
overlap = test_hashes.isin(set(train_hashes))

print("Training records:", len(df_clean))
print("Test records:", len(df_test))
print("Overlapping test records:", overlap.sum())
print(f"Test overlap percentage: {overlap.mean() * 100:.2f}%")


Training records: 175341
Test records: 82332
Overlapping test records: 8541
Test overlap percentage: 10.37%


In [30]:

# Keep unique training feature combinations
unique_train_features = df_clean[
    input_features
].drop_duplicates()

# Compare test records against training features
exact_matches = df_test[input_features].merge(
    unique_train_features,
    on=input_features,
    how="left",
    indicator=True,
    validate="many_to_one"
)

# Count exact matches
exact_overlap = (
    exact_matches["_merge"] == "both"
).sum()

print("Exact overlapping test records:", exact_overlap)

print(
    "Exact overlap percentage:",
    f"{exact_overlap / len(df_test) * 100:.2f}%"
)


Exact overlapping test records: 8541
Exact overlap percentage: 10.37%


In [31]:

# Summarize training labels for each feature combination
train_labels = (
    df_train.groupby(input_features, dropna=False)["label"]
    .agg(["min", "max"])
    .reset_index()
)

# Compare test labels with training labels
label_audit = df_test[input_features + ["label"]].merge(
    train_labels,
    on=input_features,
    how="left",
    validate="many_to_one"
)

# Identify overlapping records
matched = label_audit["min"].notna()

# Categorize label consistency
ambiguous = matched & (label_audit["min"] != label_audit["max"])

same_label = (
    matched
    & ~ambiguous
    & (label_audit["label"] == label_audit["min"])
)

opposite_label = (
    matched
    & ~ambiguous
    & (label_audit["label"] != label_audit["min"])
)

print("Overlapping test records:", matched.sum())
print("Same label:", same_label.sum())
print("Opposite label:", opposite_label.sum())
print("Conflicting training labels:", ambiguous.sum())


Overlapping test records: 8541
Same label: 8017
Opposite label: 296
Conflicting training labels: 228


In [32]:

from sklearn.model_selection import GroupShuffleSplit

# Select input features
feature_cols = df_clean.columns.drop("label").tolist()

# Group records with identical input features
groups = pd.util.hash_pandas_object(
    df_clean[feature_cols],
    index=False
)

# Create a group-aware training/validation split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(df_clean, groups=groups)
)

# Create separate datasets
df_fit = df_clean.iloc[train_idx].copy()
df_val = df_clean.iloc[val_idx].copy()

# Verify the split
shared_groups = set(groups.iloc[train_idx]) & set(groups.iloc[val_idx])

print("Training shape:", df_fit.shape)
print("Validation shape:", df_val.shape)
print("Shared feature groups:", len(shared_groups))

print("\nTraining label distribution (%):")
print(df_fit["label"].value_counts(normalize=True) * 100)

print("\nValidation label distribution (%):")
print(df_val["label"].value_counts(normalize=True) * 100)


Training shape: (140145, 43)
Validation shape: (35196, 43)
Shared feature groups: 0

Training label distribution (%):
label
1    67.978165
0    32.021835
Name: proportion, dtype: float64

Validation label distribution (%):
label
1    68.396977
0    31.603023
Name: proportion, dtype: float64


In [33]:

# Separate Features (X) and Target (y)

X_train = df_fit.drop(columns=["label"])
y_train = df_fit["label"]

X_val = df_val.drop(columns=["label"])
y_val = df_val["label"]

# Verify dimensions
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nX_val:", X_val.shape)
print("y_val:", y_val.shape)

# Verify both sets contain the same features
print(
    "\nMatching feature columns:",
    X_train.columns.equals(X_val.columns)
)


X_train: (140145, 42)
y_train: (140145,)

X_val: (35196, 42)
y_val: (35196,)

Matching feature columns: True


In [34]:

# ============================================================
# CYBERGUARD AI
# PHASE 5 — DATA CLEANING & PREPROCESSING
# Dataset: UNSW-NB15
# Task: Binary Classification (Benign vs Malicious)
# ============================================================

# 1. IMPORT LIBRARIES

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from scipy import sparse


# ============================================================
# 2. LOAD DATASETS
# ============================================================

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_path = project_root / "Data" / "Raw data"

df_train = pd.read_csv(
    data_path / "UNSW_NB15_training-set.csv"
)

df_test = pd.read_csv(
    data_path / "UNSW_NB15_testing-set.csv"
)

print("=== DATASET LOADING ===")
print("Training:", df_train.shape)
print("Testing:", df_test.shape)

assert "label" in df_train.columns
assert "label" in df_test.columns
assert set(df_train["label"].unique()) <= {0, 1}
assert set(df_test["label"].unique()) <= {0, 1}


# ============================================================
# 3. INITIAL DATA QUALITY ASSESSMENT
# ============================================================

missing_values = int(df_train.isna().sum().sum())

duplicate_count = int(
    df_train.drop(columns=["id"]).duplicated().sum()
)

print("\n=== DATA QUALITY ===")
print("Missing values:", missing_values)
print("Duplicate occurrences:", duplicate_count)

print("\nTarget distribution (%):")
print(
    df_train["label"].value_counts(
        normalize=True
    ).mul(100).round(2)
)


# ============================================================
# 4. DEFINE INPUT FEATURES
# ============================================================

excluded_columns = ["id", "attack_cat", "label"]

feature_cols = [
    col for col in df_train.columns
    if col not in excluded_columns
]

assert len(feature_cols) == 42
assert set(feature_cols).issubset(df_test.columns)

print("\n=== FEATURE SELECTION ===")
print("Input features:", len(feature_cols))
print("Excluded:", excluded_columns)


# ============================================================
# 5. INVESTIGATE CONFLICTING LABELS
# ============================================================

feature_stats = (
    df_train.groupby(feature_cols, dropna=False)["label"]
    .agg(["nunique", "size"])
)

conflicts = feature_stats[
    feature_stats["nunique"] > 1
]

conflicting_groups = len(conflicts)
conflicting_records = int(conflicts["size"].sum())

print("\n=== LABEL CONFLICTS ===")
print("Conflicting groups:", conflicting_groups)
print("Records in conflicting groups:", conflicting_records)

# Decision:
# Keep records for now.
# Conflicting labels cannot be corrected reliably
# without additional ground-truth information.


# ============================================================
# 6. AUDIT TRAIN/TEST OVERLAP
# ============================================================

train_label_summary = (
    df_train.groupby(feature_cols, dropna=False)["label"]
    .agg(["min", "max"])
    .reset_index()
)

overlap_audit = df_test[
    feature_cols + ["label"]
].merge(
    train_label_summary,
    on=feature_cols,
    how="left",
    validate="many_to_one"
)

matched = overlap_audit["min"].notna()

ambiguous = (
    matched
    & (overlap_audit["min"] != overlap_audit["max"])
)

same_label = (
    matched
    & ~ambiguous
    & (overlap_audit["label"] == overlap_audit["min"])
)

opposite_label = (
    matched
    & ~ambiguous
    & (overlap_audit["label"] != overlap_audit["min"])
)

overlap_count = int(matched.sum())
overlap_pct = overlap_count / len(df_test) * 100

print("\n=== TRAIN/TEST OVERLAP ===")
print("Overlapping test records:", overlap_count)
print(f"Overlap percentage: {overlap_pct:.2f}%")
print("Same label:", int(same_label.sum()))
print("Opposite label:", int(opposite_label.sum()))
print("Ambiguous training labels:", int(ambiguous.sum()))

# The official Test Dataset remains unchanged.
# Do not use Test results to tune preprocessing
# or model hyperparameters.


# ============================================================
# 7. CREATE WORKING DATAFRAME
# ============================================================

df_clean = df_train.drop(
    columns=["id", "attack_cat"]
).copy()

print("\n=== WORKING DATAFRAME ===")
print("Original:", df_train.shape)
print("Working copy:", df_clean.shape)

assert "label" in df_clean.columns
assert df_clean.shape[1] == 43


# ============================================================
# 8. VALIDATE NUMERICAL FEATURES
# ============================================================

numerical_features = df_clean.select_dtypes(
    include=["number"]
).columns.drop("label").tolist()

categorical_features = df_clean.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

numeric_data = df_clean[numerical_features]

infinite_count = int(
    np.isinf(numeric_data.to_numpy(dtype=float)).sum()
)

negative_count = int(
    numeric_data.lt(0).sum().sum()
)

constant_features = [
    col for col in numerical_features
    if df_clean[col].nunique() <= 1
]

print("\n=== NUMERICAL VALIDATION ===")
print("Numerical features:", len(numerical_features))
print("Infinite values:", infinite_count)
print("Negative values:", negative_count)
print("Constant features:", constant_features)

if infinite_count:
    raise ValueError("Infinite numerical values detected.")

if missing_values:
    raise ValueError("Missing values require review.")

# No automatic outlier removal:
# Extreme network traffic values may be legitimate.


# ============================================================
# 9. VALIDATE CATEGORICAL FEATURES
# ============================================================

print("\n=== CATEGORICAL VALIDATION ===")

for col in categorical_features:
    values = df_clean[col].astype("string").str.strip()

    print(
        f"{col}: {df_clean[col].nunique()} categories, "
        f"{int(values.eq('-').sum())} dash values"
    )

# Retain "-" as a distinct category in service.
# Rare categories are preserved.
# No category is automatically assumed to be erroneous.


# ============================================================
# 10. GROUP-AWARE TRAINING/VALIDATION SPLIT
# ============================================================

groups = pd.util.hash_pandas_object(
    df_clean[feature_cols],
    index=False
)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(df_clean, groups=groups)
)

df_fit = df_clean.iloc[train_idx].copy()
df_val = df_clean.iloc[val_idx].copy()

shared_groups = (
    set(groups.iloc[train_idx])
    & set(groups.iloc[val_idx])
)

assert len(shared_groups) == 0

print("\n=== TRAIN/VALIDATION SPLIT ===")
print("Training:", df_fit.shape)
print("Validation:", df_val.shape)
print("Shared feature groups:", len(shared_groups))

print("\nTraining class distribution (%):")
print(
    df_fit["label"].value_counts(
        normalize=True
    ).mul(100).round(2)
)

print("\nValidation class distribution (%):")
print(
    df_val["label"].value_counts(
        normalize=True
    ).mul(100).round(2)
)


# ============================================================
# 11. SEPARATE FEATURES AND TARGET
# ============================================================

X_train = df_fit[feature_cols].copy()
y_train = df_fit["label"].copy()

X_val = df_val[feature_cols].copy()
y_val = df_val["label"].copy()

assert X_train.columns.equals(X_val.columns)
assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)

print("\n=== FEATURES AND TARGET ===")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)


# ============================================================
# 12. BUILD PREPROCESSING PIPELINE
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            "passthrough",
            numerical_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_features
        )
    ],
    sparse_threshold=1.0
)

# Fit ONLY on Training
X_train_processed = preprocessor.fit_transform(
    X_train
)

# Transform Validation without fitting again
X_val_processed = preprocessor.transform(
    X_val
)

print("\n=== PREPROCESSING ===")
print("Training:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print(
    "Sparse matrix:",
    sparse.issparse(X_train_processed)
)


# ============================================================
# 13. VERIFY PROCESSED FEATURES
# ============================================================

processed_feature_names = (
    preprocessor.get_feature_names_out()
)

assert (
    X_train_processed.shape[1]
    == X_val_processed.shape[1]
)

assert (
    X_train_processed.shape[1]
    == len(processed_feature_names)
)

def nonfinite_count(matrix):
    values = (
        matrix.data
        if sparse.issparse(matrix)
        else np.asarray(matrix)
    )
    return int((~np.isfinite(values)).sum())

assert nonfinite_count(X_train_processed) == 0
assert nonfinite_count(X_val_processed) == 0

print("\n=== FINAL VERIFICATION ===")
print("Original input features:", len(feature_cols))
print("Encoded features:", len(processed_feature_names))
print("Training records:", X_train_processed.shape[0])
print("Validation records:", X_val_processed.shape[0])
print("Invalid processed values: 0")
print("Shared groups:", len(shared_groups))


# ============================================================
# 14. GENERATE TECHNICAL REPORT
# ============================================================

report = f"""# CyberGuard AI — Data Cleaning & Preprocessing Report

## 1. Objective

Prepare UNSW-NB15 network traffic data for supervised
Binary Classification of Benign (0) and Malicious (1) traffic.

## 2. Dataset

- Original Training Records: {len(df_train):,}
- Original Test Records: {len(df_test):,}
- Original Columns: {df_train.shape[1]}
- Input Features: {len(feature_cols)}
- Target: `label`

## 3. Data Quality Assessment

- Missing Values: {missing_values:,}
- Duplicate Occurrences (excluding id): {duplicate_count:,}
- Conflicting Feature Groups: {conflicting_groups:,}
- Records in Conflicting Groups: {conflicting_records:,}
- Infinite Numerical Values: {infinite_count}
- Negative Numerical Values: {negative_count}
- Constant Numerical Features: {len(constant_features)}

## 4. Train/Test Overlap

- Test Records with Matching Training Features: {overlap_count:,}
- Test Overlap: {overlap_pct:.2f}%
- Matching Labels: {int(same_label.sum()):,}
- Opposite Labels: {int(opposite_label.sum()):,}
- Ambiguous Training Labels: {int(ambiguous.sum()):,}

These are exact feature-value matches, not necessarily
identical real-world network connections.

The official Test Dataset was not modified.

## 5. Data Cleaning Decisions

- Excluded `id` because it is an identifier.
- Excluded `attack_cat` to avoid potential Target Leakage.
- Retained `label` as the Binary Classification target.
- Retained Duplicate Records pending further evidence.
- Retained Conflicting Labels pending ground-truth review.
- Retained `service = "-"` as a categorical value.
- Retained rare categorical values.
- Did not remove Outliers automatically.
- Did not alter the original CSV files.

## 6. Train/Validation Split

- Method: GroupShuffleSplit
- Random State: 42
- Training Records: {len(X_train):,}
- Validation Records: {len(X_val):,}
- Shared Feature Groups: {len(shared_groups)}

Identical combinations of Input Features are kept
within the same development split.

## 7. Preprocessing

- Numerical Features: {len(numerical_features)}
- Categorical Features: {", ".join(categorical_features)}
- Categorical Encoding: OneHotEncoder
- Unknown Categories: Ignored during transformation
- Numeric Transformation: Passthrough
- Encoded Feature Count: {len(processed_feature_names)}

The Preprocessor was fitted only on Training Data.
Validation Data was transformed using the fitted
Preprocessor without fitting again.

## 8. Final Outputs

- X_train_processed: {X_train_processed.shape}
- X_val_processed: {X_val_processed.shape}
- y_train: {y_train.shape}
- y_val: {y_val.shape}

## 9. Limitations and Evaluation Plan

Duplicate and conflicting records have not been
automatically removed because the available data
does not establish which observations are erroneous.

The official Test Dataset has a known feature overlap
with Training. Final Model Evaluation must explicitly
report this limitation.

Validation is used for model selection.
The official Test Dataset must not be used for
hyperparameter tuning or preprocessing decisions.

## 10. Next Phase

Model Training and Evaluation:
- DummyClassifier baseline
- RandomForestClassifier
- Precision, Recall and F1-score
- Confusion Matrix
- ROC-AUC
- Generalization and error analysis
"""

report_path = (
    project_root
    / "docs"
    / "data-cleaning-report.md"
)

report_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

if report_path.exists():
    print(
        "\nReport already exists; "
        "existing documentation was not overwritten."
    )
else:
    report_path.write_text(
        report,
        encoding="utf-8"
    )
    print("\nReport created:", report_path)


# ============================================================
# PHASE 5 COMPLETE
# ============================================================

print("\n" + "=" * 55)
print("CYBERGUARD AI — PHASE 5 COMPLETE")
print("=" * 55)

print("Data Quality Assessment: Complete")
print("Feature Selection: Complete")
print("Train/Validation Split: Complete")
print("Categorical Encoding: Complete")
print("Preprocessing Verification: Complete")
print("Technical Report: Prepared")

print("\nReady for Phase 6 — Model Training.")


=== DATASET LOADING ===
Training: (175341, 45)
Testing: (82332, 45)

=== DATA QUALITY ===
Missing values: 0
Duplicate occurrences: 67601

Target distribution (%):
label
1    68.06
0    31.94
Name: proportion, dtype: float64

=== FEATURE SELECTION ===
Input features: 42
Excluded: ['id', 'attack_cat', 'label']

=== LABEL CONFLICTS ===
Conflicting groups: 229
Records in conflicting groups: 940

=== TRAIN/TEST OVERLAP ===
Overlapping test records: 8541
Overlap percentage: 10.37%
Same label: 8017
Opposite label: 296
Ambiguous training labels: 228

=== WORKING DATAFRAME ===
Original: (175341, 45)
Working copy: (175341, 43)

=== NUMERICAL VALIDATION ===
Numerical features: 39
Infinite values: 0
Negative values: 0
Constant features: []

=== CATEGORICAL VALIDATION ===
proto: 133 categories, 0 dash values
service: 13 categories, 94168 dash values
state: 9 categories, 0 dash values

=== TRAIN/VALIDATION SPLIT ===
Training: (140145, 43)
Validation: (35196, 43)
Shared feature groups: 0

Training cl